# RateMyProfessor Analyzer
**Author:** Rayan Khan
**GitHub:** https://github.com/Rayan-K-Khan
**Date:** June 2026

In [ ]:
!pip install requests beautifulsoup4
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import re
import json
import base64
from collections import Counter
import statistics

#Search and Scrape

In [ ]:
def search_professor(prof_name, school_name):
    url = "https://www.ratemyprofessors.com/graphql"

    headers = {
        "Authorization": "Basic dGVzdDp0ZXN0",
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0"
    }

    query = """
    query NewSearchTeachersQuery($query: TeacherSearchQuery!) {
      newSearch {
        teachers(query: $query, first: 10) {
          edges {
            node {
              id
              legacyId
              firstName
              lastName
              department
              school {
                name
                city
                state
              }
              avgRating
              numRatings
            }
          }
        }
      }
    }
    """

    payload = {
        "query": query,
        "variables": {
            "query": {
                "text": prof_name
            }
        }
    }

    response = requests.post(url, json=payload, headers=headers)
    teachers = response.json()["data"]["newSearch"]["teachers"]["edges"]

    filtered = [t for t in teachers if school_name.lower() in t["node"]["school"]["name"].lower()]

    if not filtered:
        print("No professors found at that school, showing all results:")
        filtered = teachers

    if not filtered:
        print("No professors found.")
        return None

    print("\nResults found:")
    for i, t in enumerate(filtered):
        node = t["node"]
        print(f"{i+1}. {node['firstName']} {node['lastName']} — {node['department']} at {node['school']['name']} ({node['school']['city']}, {node['school']['state']}) | Rating: {node['avgRating']}")

    choice = int(input("\nEnter the number of the correct professor: ")) - 1
    return filtered[choice]["node"]

prof_name = input("Enter professor name: ")
school_name = input("Enter school name: ")
result = search_professor(prof_name, school_name)
print(result)

In [ ]:
def safe_extract(pattern, text, default="0"):
    """Safely extracts a regex match. Returns default if no match or if value is null."""
    match = re.search(pattern, text)
    if match:
        val = match.group(1).strip()
        if val in ["null", "None", "", "null;"]:
            return default
        return val
    return default

def scrape_professor(url, prof_id):
    page = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    idx = page.text.find(f'\"legacyId\":{prof_id}')
    chunk = page.text[max(0, idx-50):idx+5000]
    professor = {
        "name": safe_extract(r'\"firstName\":\"(.*?)\"', chunk, "Unknown") + " " + safe_extract(r'\"lastName\":\"(.*?)\"', chunk, "Professor"),
        "department": safe_extract(r'\"department\":\"(.*?)\"', chunk, "Unknown"),
        "avg_rating": float(safe_extract(r'\"avgRating\":([^,}]*)', chunk, "0.0")),
        "avg_difficulty": float(safe_extract(r'\"avgDifficulty\":([^,}]*)', chunk, "0.0")),
        "num_ratings": int(safe_extract(r'\"numRatings\":([^,}]*)', chunk, "0")),
        "would_take_again": float(safe_extract(r'\"wouldTakeAgainPercent\":([^,}]*)', chunk, "0.0")),
        "ratings_distribution": {
            "1 star": int(safe_extract(r'\"r1\":([^,}]*)', chunk, "0")),
            "2 star": int(safe_extract(r'\"r2\":([^,}]*)', chunk, "0")),
            "3 star": int(safe_extract(r'\"r3\":([^,}]*)', chunk, "0")),
            "4 star": int(safe_extract(r'\"r4\":([^,}]*)', chunk, "0")),
            "5 star": int(safe_extract(r'\"r5\":([^,}]*)', chunk, "0")),
        }
    }

    encoded_id = base64.b64encode(f"Teacher-{prof_id}".encode()).decode()
    api_response = requests.post(
        "https://www.ratemyprofessors.com/graphql",
        headers={
            "Authorization": "Basic dGVzdDp0ZXN0",
            "Content-Type": "application/json",
            "User-Agent": "Mozilla/5.0",
            "Referer": url
        },
        json={
            "query": """
            query RatingsListQuery($id: ID!) {
              node(id: $id) {
                ... on Teacher {
                  ratings(first: 100) {
                    edges {
                      node {
                        date
                        class
                        comment
                        helpfulRating
                        difficultyRating
                        grade
                        wouldTakeAgain
                      }
                    }
                  }
                }
              }
            }
            """,
            "variables": {"id": encoded_id}
        }
    )

    try:
        res_json = api_response.json()
        node = res_json.get("data", {}).get("node", {})
        if node and "ratings" in node and node["ratings"]:
            edges = node["ratings"].get("edges", [])
        else:
            edges = []
    except Exception:
        edges = []

    reviews = [edge["node"] for edge in edges if edge and "node" in edge]
    return professor, reviews

#Load ratings and data setup

In [ ]:
if result:
    numeric_id = result["legacyId"]
    rmp_url = f"https://www.ratemyprofessors.com/professor/{numeric_id}"
    professor, reviews = scrape_professor(rmp_url, numeric_id)
    all_comments = "\n".join([f"- {r['comment']}" for r in reviews if r['comment'].strip()])
    difficulty_ratings = [r["difficultyRating"] for r in reviews]
    print(f"Loaded: {professor['name']} — {professor['num_ratings']} ratings")

In [ ]:
url = "https://www.ratemyprofessors.com/graphql"

headers = {
    "Authorization": "Basic dGVzdDp0ZXN0",
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0",
    "Referer": f"https://www.ratemyprofessors.com/professor/{result['legacyId']}"
}

query = """
query RatingsListQuery($id: ID!) {
  node(id: $id) {
    ... on Teacher {
      ratings(first: 100) {
        edges {
          node {
            date
            class
            comment
            helpfulRating
            difficultyRating
            grade
            wouldTakeAgain
          }
        }
      }
    }
  }
}
"""

payload = {
    "query": query,
    "variables": {
        "id": result["id"]
    }
}

response = requests.post(url, json=payload, headers=headers)
print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    print(json.dumps(response.json(), indent=2))

#Difficulty Distribution

In [ ]:
data = response.json()
edges = data["data"]["node"]["ratings"]["edges"]

reviews = [edge["node"] for edge in edges]

difficulty_ratings = [r["difficultyRating"] for r in reviews]

print(difficulty_ratings)
print(f"Total reviews: {len(difficulty_ratings)}")

colors = ["green", "lightgreen", "yellow", "orange", "red"]

diff_counts = Counter(difficulty_ratings)
sns.barplot(x=[1,2,3,4,5], y=[diff_counts.get(i, 0) for i in range(1,6)], palette=colors)
plt.title(f"Difficulty Ratings for {professor['name']}")
plt.xlabel("Difficulty (Easy to Hard)")
plt.ylabel("Frequency")
plt.show()

#Quality Distribution

In [ ]:
ratings = list(professor["ratings_distribution"].values())
print(ratings)

sns.barplot(x=[1,2,3,4,5], y=ratings, palette=colors[::-1])
plt.title(f'Quality Ratings for {professor["name"]}')
plt.xlabel('Rating (Worst to Best)')
plt.ylabel('Frequency')
plt.show()



In [ ]:
courses = [r["class"] for r in reviews if r["class"].strip()]
course_counts = Counter(courses)
num_reviews = 0

print("Courses taught:")
for course, count in course_counts.most_common():
    print(f"  {course}: {count} reviews")
    num_reviews += count

print(f"\nTotal reviews: {num_reviews}")

#Courses taught

In [ ]:
courses_sorted = course_counts.most_common()
course_names = [c[0] for c in courses_sorted]
course_review_counts = [c[1] for c in courses_sorted]

sns.barplot(x=course_names, y=course_review_counts)
plt.title(f"Courses Taught by {professor['name']}")
plt.xlabel("Course")
plt.ylabel("Number of Reviews")
plt.show()

#Overall letter grade distribution

In [ ]:
grade_list = [
    r["grade"].strip() for r in reviews
    if r.get("grade") and r["grade"].strip() not in ["", "N/A", "Select", "Not given"]
]

if not grade_list:
    display(Markdown("### Grade Distribution"))
    display(Markdown("*No student letter grades were submitted for this professor.*"))

else:
    grade_counts = Counter(grade_list)

    standard_order = ['A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D+', 'D', 'D-', 'F', 'WD', 'INC']

    ordered_grades = [g for g in standard_order if g in grade_counts]

    extra_grades = sorted([g for g in grade_counts if g not in standard_order])
    final_x_order = ordered_grades + extra_grades

    ordered_y_counts = [grade_counts[g] for g in final_x_order]

    fig, ax = plt.subplots(figsize=(10, 5))

    palette = sns.color_palette("plasma", len(final_x_order))

    sns.barplot(x=final_x_order, y=ordered_y_counts, ax=ax, palette=palette)

    ax.set_title(f"Reported Letter Grade Distribution for {professor.get('name', 'Professor')}", fontsize=14, weight='bold', pad=15)
    ax.set_xlabel("Grade Received", fontsize=12, weight='bold', labelpad=10)
    ax.set_ylabel("Number of Students", fontsize=12, weight='bold', labelpad=10)

    ax.yaxis.get_major_locator().set_params(integer=True)

    for i, count in enumerate(ordered_y_counts):
        ax.text(i, count + 0.05, str(count), ha='center', va='bottom', fontsize=11, weight='bold')

    plt.show()

#Course-by-course letter grade distribution

In [ ]:
course_grade_data = []
for r in reviews:
    course_code = r.get("class", "").strip().upper()
    grade_received = r.get("grade", "").strip()

    invalid_tags = ["", "N/A", "Select", "Not given", "NONE"]
    if course_code and grade_received and grade_received not in invalid_tags:
        course_grade_data.append({"Course": course_code, "Grade": grade_received})

if not course_grade_data:
    display(Markdown("###Course-by-Course Grade Distribution"))
    display(Markdown("*No course-specific grade data is available for this professor.*"))

else:
    df = pd.DataFrame(course_grade_data)
    unique_courses = sorted(df["Course"].unique())
    num_courses = len(unique_courses)

    df_summary = df.groupby(["Course", "Grade"]).size().reset_index(name="Student Count")
    df_summary.to_csv("course_grade_summary.csv", index=False)

    standard_scale = ['A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D+', 'D', 'D-', 'F', 'WD', 'INC']

    cols = 2 if num_courses > 1 else 1
    rows = (num_courses + 1) // 2 if num_courses > 1 else 1

    fig, axes = plt.subplots(rows, cols, figsize=(14, 4.8 * rows), squeeze=False)
    axes_flat = axes.flatten()

    for idx, course_title in enumerate(unique_courses):
        ax = axes_flat[idx]

        df_subset = df[df["Course"] == course_title]
        counts = df_subset["Grade"].value_counts()

        sorted_x = [g for g in standard_scale if g in counts]
        outlier_entries = sorted([g for g in counts.index if g not in standard_scale])
        final_x_axis = sorted_x + outlier_entries
        final_y_counts = [counts[g] for g in final_x_axis]

        sns.barplot(x=final_x_axis, y=final_y_counts, ax=ax, palette=sns.color_palette("viridis", len(final_x_axis)))

        ax.set_title(f"Course: {course_title} (n = {len(df_subset)} reviews)", fontsize=12, weight='bold', pad=10)
        ax.set_xlabel("Grade Earned", fontsize=10, weight='bold')
        ax.set_ylabel("Number of Students", fontsize=10, weight='bold')
        ax.yaxis.get_major_locator().set_params(integer=True)

        for i, val in enumerate(final_y_counts):
            ax.text(i, val + 0.05, str(val), ha='center', va='bottom', fontsize=10, weight='bold')

    for extra_idx in range(num_courses, len(axes_flat)):
        fig.delaxes(axes_flat[extra_idx])

    plt.tight_layout()

    plt.savefig('course_grade_distributions.png', bbox_inches='tight', dpi=300)

    display(Markdown("###Course-by-Course Grade Distribution Summary"))
    display(Markdown(f"Successfully generated a multi-panel visual grid tracking **{num_courses} unique courses**."))

#Retake percentage

In [ ]:
wta_percent = professor.get("would_take_again", 0.0)
count = professor.get("num_ratings", 0)

if count == 0:
    display(Markdown("###Retake Percentage Visualization"))
    display(Markdown("*No rating data available to display 'Would Take Again' percentage.*"))

else:
    labels = ['Would Take Again', 'Would Not']
    sizes = [wta_percent, max(0.0, 100.0 - wta_percent)]

    colors = ['#01c61f', '#ff0020']

    explode = (0.05, 0)
    plt.figure(figsize=(6, 6))

    patches, texts, autotexts = plt.pie(
        sizes,
        explode=explode,
        labels=labels,
        colors=colors,
        autopct='%1.1f%%',
        startangle=140,
        textprops={'fontsize': 12}
    )

    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_weight('bold')
        autotext.set_fontsize(11)

    for text in texts:
        text.set_weight('bold')

    plt.axis('equal')

    plt.title(f"Percentage of Students Who Would Take\n{professor['name']} Again", fontsize=14, pad=20, weight='bold')

    plt.show()

#Descriptive stats

In [ ]:
quality_ratings = [r["helpfulRating"] for r in reviews if r.get("helpfulRating") is not None]
difficulty_ratings = [r["difficultyRating"] for r in reviews if r.get("difficultyRating") is not None]

if len(quality_ratings) == 0 or len(difficulty_ratings) == 0:
    display(Markdown("### Descriptive Statistics"))
    display(Markdown("*No numeric review data available to calculate statistics.*"))

else:
    qual_mean = np.mean(quality_ratings)
    qual_median = np.median(quality_ratings)
    qual_std = np.std(quality_ratings)
    try:
        qual_mode = statistics.mode(quality_ratings)
    except Exception:
        qual_mode = "Multimodal"

    diff_mean = np.mean(difficulty_ratings)
    diff_median = np.median(difficulty_ratings)
    diff_std = np.std(difficulty_ratings)
    try:
        diff_mode = statistics.mode(difficulty_ratings)
    except Exception:
        diff_mode = "Multimodal"
    stats_data = {
        "Quality Rating": [
            f"{qual_mean:.2f} / 5.0",
            f"{qual_median:.1f}",
            f"{qual_mode}",
            f"{qual_std:.2f}"
        ],
        "Difficulty Rating": [
            f"{diff_mean:.2f} / 5.0",
            f"{diff_median:.1f}",
            f"{diff_mode}",
            f"{diff_std:.2f}"
        ]
    }

    metrics_labels = [
        "Mean",
        "Median",
        "Mode",
        "Standard Deviation"
    ]

    df_stats = pd.DataFrame(stats_data, index=metrics_labels)

    display(Markdown("###Descriptive Statistics"))
    display(df_stats)

    display(Markdown("*Note: A lower Standard Deviation (<0.5) means students generally agree on the rating, while a higher Standard Deviation (>1) means opinions are highly polarized.*"))

#AI generated summary (Groq API)

In [ ]:
from google.colab import userdata
API_key = userdata.get('RMP')

In [ ]:
response = requests.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {API_key}",
        "Content-Type": "application/json"
    },
    json={
        "model": "llama-3.3-70b-versatile",
        "messages": [
            {
                "role": "user",
                "content": f"""Based on these student reviews for professor {professor['name']},
write a concise 3-4 sentence summary covering:
- Teaching style and effectiveness
- Exam and grading difficulty
- Whether students recommend them

Reviews:
{all_comments}"""
            }
        ]
    }
)

summary = response.json()["choices"][0]["message"]["content"]

In [ ]:
from IPython.display import display, Markdown

if num_reviews==0:
  print("This professor has no available reviews at this time.")
elif num_reviews<5:
  display(Markdown(summary))
  print(f"\033[91mWARNING: This professor only has {num_reviews} review(s). This limited number of reviews may not provide a holistic rating. Proceed with caution.\033[0m")
else:
  display(Markdown(summary))